# 🦷 GigiNet AI — FASE 2: Transfer Learning
## Tambah Kelas: Calculus, Gingivitis, ToothDiscoloration, Ulcers

---
### 📋 Cara Pakai:
1. Ganti Runtime ke **T4 GPU** (Runtime → Change runtime type → T4 GPU)
2. Jalankan sel satu per satu dari atas ke bawah
3. Ikuti instruksi di setiap sel

---

## ✅ SEL 1 — Cek GPU & Install Paket

In [ ]:
# Cek GPU
!nvidia-smi

# Install paket
!pip install ultralytics -q

print('\n✅ GPU siap dan paket terinstall!')

## ✅ SEL 2 — Upload Model Fase 1 (best.pt)

> **File yang diupload:** `yolo_dental_detection.pt` yang ada di folder `C:\Users\jarvi\websitegigi\backend\models\` di laptop Anda

In [ ]:
from google.colab import files
import os, shutil

print('📤 Silakan upload file model Fase 1 Anda...')
print('   File: yolo_dental_detection.pt (dari folder backend/models/)')
uploaded = files.upload()

# Simpan dengan nama standar
for fname in uploaded.keys():
    shutil.copy(fname, '/content/fase1_best.pt')
    print(f'✅ Model Fase 1 disimpan sebagai: /content/fase1_best.pt')

## ✅ SEL 3 — Upload & Download Dataset dari Kaggle

### Cara dapat file `kaggle.json`:
1. Buka **[kaggle.com](https://kaggle.com)**
2. Klik foto profil kanan atas → **Settings**
3. Scroll ke bagian **API** → Klik **"Create New Token"**
4. File `kaggle.json` akan otomatis terdownload ke laptop Anda
5. Upload file itu di sel ini

In [ ]:
from google.colab import files
import os

print('📤 Upload file kaggle.json Anda...')
uploaded = files.upload()  # pilih file kaggle.json

# Setup Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('✅ Kaggle API siap!')
print()
print('📥 Mendownload dataset Oral Diseases...')

# Download dataset dari Kaggle
!kaggle datasets download -d salmansajid05/oral-diseases --path /content/ -q

print('📦 Mengekstrak dataset...')
!unzip -q /content/oral-diseases.zip -d /content/Dataset_Oral

print('\n✅ Dataset berhasil didownload!')
print('\n📂 Struktur folder dataset:')
!ls /content/Dataset_Oral/

## ✅ SEL 4 — Cek Isi Dataset & Kelas yang Tersedia

In [ ]:
import os

base = '/content/Dataset_Oral'

# Tampilkan semua folder
print('📁 Isi folder dataset:')
for root, dirs, files_list in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 2:
        break
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    if level >= 1:
        print(f'{indent}   ({len(files_list)} file)')

# Cari data.yaml
print('\n🔍 Mencari file data.yaml...')
for root, dirs, files_list in os.walk(base):
    for f in files_list:
        if f == 'data.yaml':
            yaml_path = os.path.join(root, f)
            print(f'\n✅ Ditemukan: {yaml_path}')
            print('\n📄 Isi data.yaml:')
            with open(yaml_path, 'r') as yf:
                print(yf.read())

## ✅ SEL 5 — Siapkan data.yaml untuk Training

> **Catatan:** Sel ini akan otomatis membuat `data.yaml` yang benar berdasarkan struktur dataset

In [ ]:
import os, yaml, glob

base = '/content/Dataset_Oral'

# Cari path train, valid, test secara otomatis
def find_folder(base, names):
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            if d.lower() in names:
                full = os.path.join(root, d, 'images')
                if os.path.exists(full):
                    return full
    return None

train_path = find_folder(base, ['train', 'training'])
val_path   = find_folder(base, ['val', 'valid', 'validation'])
test_path  = find_folder(base, ['test', 'testing'])

print(f'Train : {train_path}')
print(f'Val   : {val_path}')
print(f'Test  : {test_path}')

# Kelas penyakit gigi yang akan dideteksi
# Sesuaikan dengan isi data.yaml yang ditampilkan di SEL 4!
class_names = {
    0: 'Calculus',
    1: 'Caries',
    2: 'Gingivitis',
    3: 'Tooth Discoloration',
    4: 'Ulcers',
}

# Buat data.yaml baru
data_config = {
    'train': train_path or f'{base}/train/images',
    'val':   val_path   or f'{base}/valid/images',
    'test':  test_path  or f'{base}/test/images',
    'nc': len(class_names),
    'names': list(class_names.values())
}

yaml_out = '/content/data_fase2.yaml'
with open(yaml_out, 'w') as f:
    yaml.dump(data_config, f, allow_unicode=True, default_flow_style=False)

print(f'\n✅ data.yaml siap di: {yaml_out}')
print('\n📄 Isi:')
with open(yaml_out) as f:
    print(f.read())

## 🚀 SEL 6 — MULAI TRAINING FASE 2!

> ⏱️ Estimasi waktu: **30-60 menit** dengan T4 GPU (jauh lebih cepat dari Fase 1 karena Transfer Learning!)

In [ ]:
from ultralytics import YOLO

print('====== MEMULAI PELATIHAN AI - FASE 2 ======')
print('📌 Model awal : fase1_best.pt (Transfer Learning)')
print('📌 Dataset    : Oral Diseases (Calculus, Caries, Gingivitis, dll)')
print('📌 Epoch      : 100')
print('===========================================')

# Load model dari Fase 1 sebagai titik awal!
model = YOLO('/content/fase1_best.pt')

results = model.train(
    data='/content/data_fase2.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='GigiNet_Fase2',
    project='/content/runs',
    device=0,
    patience=20,      # Berhenti otomatis jika tidak ada peningkatan 20 epoch
    lr0=0.001,        # Learning rate kecil untuk fine-tuning
    lrf=0.01,
    warmup_epochs=3,
    exist_ok=True,
    save=True,
    plots=True,
)

print('\n🎉 TRAINING SELESAI!')

## ✅ SEL 7 — Lihat Hasil Evaluasi

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob

model = YOLO('/content/runs/GigiNet_Fase2/weights/best.pt')
metrics = model.val(data='/content/data_fase2.yaml')

print('\n📊 HASIL EVALUASI MODEL FASE 2:')
print(f'mAP50    : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall   : {metrics.box.mr:.4f}')

print('\n📈 mAP per kelas penyakit:')
for i, ap in enumerate(metrics.box.maps):
    print(f'  {model.names[i]:25s}: {ap*100:.1f}%')

# Tampilkan grafik training
results_img = '/content/runs/GigiNet_Fase2/results.png'
if os.path.exists(results_img):
    print('\n📉 Grafik Training:')
    display(Image(results_img))

## ✅ SEL 8 — Uji Coba dengan Foto Gigi Asli

In [ ]:
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display
import os

model = YOLO('/content/runs/GigiNet_Fase2/weights/best.pt')

print('📤 Upload foto gigi untuk dicoba...')
uploaded = files.upload()

for fname in uploaded.keys():
    results = model(fname, conf=0.25, save=True, project='/content', name='test_hasil')
    
    print(f'\n🔍 Hasil deteksi pada: {fname}')
    for r in results:
        if len(r.boxes) == 0:
            print('  ✅ Tidak ada penyakit terdeteksi (Gigi Sehat)')
        for box in r.boxes:
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            print(f'  🦷 {model.names[cls]:25s} | Kepercayaan AI: {conf*100:.1f}%')

# Tampilkan gambar hasil anotasi
hasil_imgs = sorted(os.listdir('/content/test_hasil'))
for img in hasil_imgs:
    if img.endswith(('.jpg', '.png', '.jpeg')):
        display(Image(f'/content/test_hasil/{img}', width=600))

## ✅ SEL 9 — Download Model Baru ke Laptop

> Setelah download, **rename** file menjadi `yolo_dental_detection.pt` dan taruh di `C:\Users\jarvi\websitegigi\backend\models\`

In [ ]:
from google.colab import files
import shutil

src  = '/content/runs/GigiNet_Fase2/weights/best.pt'
dest = '/content/GigiNet_v2_best.pt'

shutil.copy(src, dest)

print('📥 Mendownload model GigiNet AI Fase 2...')
print()
print('✅ Setelah download selesai:')
print('   1. Rename file: GigiNet_v2_best.pt → yolo_dental_detection.pt')
print('   2. Copy ke folder: C:\\Users\\jarvi\\websitegigi\\backend\\models\\')
print('   3. Server backend otomatis reload & pakai model baru!')
print()

files.download(dest)
print('🎉 Download selesai!')